In [ ]:
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType
import reverse_geocoder as rg
from datetime import date, timedelta

In [ ]:
# Remove this line before running Data Factory Pipeline
start_date = date.today() - timedelta(7)

In [ ]:
df = spark.read.table("earthquake_events_silver").filter(col("time") > start_date)

In [ ]:
def get_country_code(lat, lon):
    """
    Retrieve the country code for a given latitude & longitude.

    Parameters:
    lat (float or str): Latitude of the location.
    lon (float or str): Longitude of the location.

    Returns:
    str: Country code of the location, retrieved using reverse geocoder API.
    """
    coordinates = (float(lat), float(lon))
    return rg.search(coordinates)[0].get("cc")

In [ ]:
# Registering udf so they can be used on spark dataframes
get_country_code_udf = udf(get_country_code, StringType()) 

In [ ]:
# Adding country code and city attributes
df_with_location = \
                df.\
                    withColumn("country_code", get_country_code_udf(col("latitude"), col("longitude")))

In [ ]:
# Adding significance classification
df_with_location_sig_class = \
                            df_with_location.\
                                withColumn("sig_class",
                                            when(col("sig") < 100, "Low").\
                                            when((col("sig") >= 100) & (col("sig") < 500), "Moderate").\
                                            otherwise("High")
                                            )                                         

In [ ]:
# Appending to the gold table
df_with_location_sig_class.write.mode("append").saveAsTable("earthquake_events_gold")